# 13 — Relationship Analysis

## Sales Order, Backlog & Inventory Analytics

### Objective

Analyze the relationships between major operational and commercial performance indicators across the business.

Rather than analyzing each business area independently, this notebook connects multiple operational dimensions to understand how performance in one area is associated with outcomes in another.

### Analysis Scope

The relationship analysis will investigate:

- Inventory availability vs backlog
- Inventory coverage vs fulfillment performance
- Stock status vs product backlog
- Reorder pressure vs backlog exposure
- Product demand vs inventory position
- Product sales vs backlog exposure
- Fulfillment rate vs delivery performance
- Backlog vs delivery delay
- Order processing time vs delivery performance
- Supplier reliability vs procurement delay
- Supplier delay vs inventory risk
- Procurement availability vs replenishment pressure
- Product/category operational relationships
- Warehouse-level operational relationships
- Correlation analysis across key KPIs

### Analytical Goal

The goal is to identify operational relationships that may explain where business performance deteriorates and determine which factors are most strongly associated with:

- Backlog
- Stock shortages
- Poor fulfillment
- Delivery delays
- OTIF failure
- Replenishment pressure

The analysis focuses on **relationships and associations** rather than assuming that correlation proves causation.

In [1]:
# Import required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("Pandas version :", pd.__version__)
print("NumPy version  :", np.__version__)

print(
    "\nRelationship Analysis environment initialized successfully."
)

Pandas version : 3.0.6
NumPy version  : 2.5.3

Relationship Analysis environment initialized successfully.


## Step 2 — Define Feature-Engineered Data Path

Define the project directory and feature-engineered data location.

The Relationship Analysis will use the same validated analytical datasets from the completed project stages to ensure consistency across Sales, Backlog, Inventory, Procurement, Shipment, and Fulfillment metrics.

In [2]:
# Define project paths

project_path = Path.cwd().parent

featured_data_path = (
    project_path / "data" / "featured"
)


print("Project Path       :", project_path)
print("Featured Data Path :", featured_data_path)

print(
    "\nFeatured Data Folder Exists :",
    featured_data_path.exists()
)

Project Path       : c:\Users\Admin\OneDrive\Documents\Sales_Order_Backlog_Inventory_Analytics
Featured Data Path : c:\Users\Admin\OneDrive\Documents\Sales_Order_Backlog_Inventory_Analytics\data\featured

Featured Data Folder Exists : True


## Step 3 — Load Core Relationship Analysis Datasets

Load the feature-engineered datasets required for cross-functional relationship analysis.

The analysis will use:

- Sales Orders
- Sales Order Items
- Inventory
- Inventory Transactions
- Shipments
- Purchase Orders
- Products
- Suppliers
- Warehouses

These datasets provide the foundation for analyzing relationships between demand, inventory, backlog, fulfillment, delivery, and procurement performance.

In [3]:
# Load core feature-engineered datasets

sales_orders = pd.read_csv(
    featured_data_path / "sales_orders_featured.csv"
)

sales_order_items = pd.read_csv(
    featured_data_path / "sales_order_items_featured.csv"
)

inventory = pd.read_csv(
    featured_data_path / "inventory_featured.csv"
)

inventory_transactions = pd.read_csv(
    featured_data_path / "inventory_transactions_featured.csv"
)

shipments = pd.read_csv(
    featured_data_path / "shipments_featured.csv"
)

purchase_orders = pd.read_csv(
    featured_data_path / "purchase_orders_featured.csv"
)

products = pd.read_csv(
    featured_data_path / "products_featured.csv"
)

suppliers = pd.read_csv(
    featured_data_path / "suppliers_featured.csv"
)

warehouses = pd.read_csv(
    featured_data_path / "warehouses_featured.csv"
)


relationship_datasets = {
    "Sales Orders": sales_orders,
    "Sales Order Items": sales_order_items,
    "Inventory": inventory,
    "Inventory Transactions": inventory_transactions,
    "Shipments": shipments,
    "Purchase Orders": purchase_orders,
    "Products": products,
    "Suppliers": suppliers,
    "Warehouses": warehouses
}


print("Relationship Analysis datasets loaded successfully.\n")


for name, df in relationship_datasets.items():

    print(
        f"{name:<25} "
        f"{df.shape[0]:>7,} rows × "
        f"{df.shape[1]:>2} columns"
    )

Relationship Analysis datasets loaded successfully.

Sales Orders               25,000 rows × 26 columns
Sales Order Items          61,397 rows × 20 columns
Inventory                   3,000 rows × 15 columns
Inventory Transactions    110,000 rows ×  6 columns
Shipments                  24,798 rows × 21 columns
Purchase Orders             8,500 rows × 23 columns
Products                      500 rows × 10 columns
Suppliers                     100 rows ×  7 columns
Warehouses                      6 rows ×  5 columns


## Step 4 — Validate Required Analysis Fields

Validate that all fields required for cross-functional relationship analysis are available in the loaded datasets.

This prevents downstream joins, aggregations, and statistical analysis from failing because of missing analytical variables.

In [4]:
# Define required fields for relationship analysis

required_fields = {

    "Sales Orders": [
        "order_id",
        "customer_id",
        "warehouse_id",
        "ordered_qty",
        "fulfilled_qty",
        "backlog_qty",
        "order_value",
        "backlog_value",
        "order_fill_rate_pct",
        "priority"
    ],

    "Sales Order Items": [
        "order_item_id",
        "order_id",
        "product_id",
        "warehouse_id",
        "ordered_qty",
        "fulfilled_qty",
        "backlog_qty",
        "backlog_value",
        "line_fill_rate_pct"
    ],

    "Inventory": [
        "product_id",
        "warehouse_id",
        "avg_daily_demand",
        "safety_stock",
        "reorder_point",
        "on_hand_qty",
        "allocated_qty",
        "available_qty",
        "inventory_value",
        "inventory_days_on_hand",
        "reorder_required_flag"
    ],

    "Shipments": [
        "order_id",
        "warehouse_id",
        "chronology_valid_flag",
        "delivery_delay_days_clean",
        "on_time_flag_clean",
        "otif_flag_clean"
    ],

    "Purchase Orders": [
        "po_id",
        "product_id",
        "warehouse_id",
        "supplier_id",
        "ordered_qty",
        "received_qty",
        "supplier_delay_days"
    ],

    "Products": [
        "product_id",
        "supplier_id",
        "category",
        "unit_cost",
        "unit_price"
    ],

    "Suppliers": [
        "supplier_id",
        "contract_lead_time_days",
        "quality_rating",
        "reliability_factor"
    ],

    "Warehouses": [
        "warehouse_id",
        "warehouse_name",
        "region"
    ]
}


dataset_lookup = {
    "Sales Orders": sales_orders,
    "Sales Order Items": sales_order_items,
    "Inventory": inventory,
    "Shipments": shipments,
    "Purchase Orders": purchase_orders,
    "Products": products,
    "Suppliers": suppliers,
    "Warehouses": warehouses
}


all_required_fields_present = True


print("REQUIRED FIELD VALIDATION\n")


for name, fields in required_fields.items():

    df = dataset_lookup[name]

    missing_fields = [
        column
        for column in fields
        if column not in df.columns
    ]

    if missing_fields:

        all_required_fields_present = False

        print(
            f"{name:<20} : FAIL "
            f"— Missing {missing_fields}"
        )

    else:

        print(
            f"{name:<20} : PASS"
        )


print(
    "\nREQUIRED FIELD VALIDATION STATUS :",
    "PASSED"
    if all_required_fields_present
    else "REVIEW REQUIRED"
)

REQUIRED FIELD VALIDATION

Sales Orders         : PASS
Sales Order Items    : PASS
Inventory            : PASS
Shipments            : PASS
Purchase Orders      : PASS
Products             : PASS
Suppliers            : PASS
Warehouses           : PASS

REQUIRED FIELD VALIDATION STATUS : PASSED


## Step 5 — Build Product-Warehouse Fulfillment Summary

Aggregate sales order item performance to the Product + Warehouse level.

This creates a common analytical grain with the inventory dataset and allows inventory availability to be compared directly with:

- Ordered quantity
- Fulfilled quantity
- Backlog quantity
- Backlog value
- Quantity fill rate
- Backlog rate
- Number of order lines
- Number of backlogged order lines

This table will form the foundation for inventory-to-fulfillment relationship analysis.

In [5]:
# Build Product-Warehouse fulfillment summary

product_warehouse_fulfillment = (
    sales_order_items
    .groupby(
        ["product_id", "warehouse_id"],
        as_index=False
    )
    .agg(
        order_lines=("order_item_id", "count"),
        ordered_qty=("ordered_qty", "sum"),
        fulfilled_qty=("fulfilled_qty", "sum"),
        backlog_qty=("backlog_qty", "sum"),
        backlog_value=("backlog_value", "sum")
    )
)


# Calculate derived fulfillment KPIs

product_warehouse_fulfillment[
    "qty_fill_rate_pct"
] = np.where(
    product_warehouse_fulfillment["ordered_qty"] > 0,
    (
        product_warehouse_fulfillment["fulfilled_qty"]
        / product_warehouse_fulfillment["ordered_qty"]
    ) * 100,
    np.nan
)


product_warehouse_fulfillment[
    "backlog_rate_pct"
] = np.where(
    product_warehouse_fulfillment["ordered_qty"] > 0,
    (
        product_warehouse_fulfillment["backlog_qty"]
        / product_warehouse_fulfillment["ordered_qty"]
    ) * 100,
    np.nan
)


backlogged_line_counts = (
    sales_order_items
    .assign(
        backlogged_line=lambda x:
        (x["backlog_qty"] > 0).astype(int)
    )
    .groupby(
        ["product_id", "warehouse_id"]
    )["backlogged_line"]
    .sum()
    .rename("backlogged_lines")
    .reset_index()
)


product_warehouse_fulfillment = (
    product_warehouse_fulfillment
    .merge(
        backlogged_line_counts,
        on=["product_id", "warehouse_id"],
        how="left",
        validate="one_to_one"
    )
)


print("PRODUCT-WAREHOUSE FULFILLMENT SUMMARY\n")

print(
    "Records                 :",
    f"{len(product_warehouse_fulfillment):,}"
)

print(
    "Unique Products         :",
    product_warehouse_fulfillment[
        "product_id"
    ].nunique()
)

print(
    "Unique Warehouses       :",
    product_warehouse_fulfillment[
        "warehouse_id"
    ].nunique()
)

print(
    "Total Order Lines       :",
    f"{product_warehouse_fulfillment['order_lines'].sum():,}"
)

print(
    "Total Ordered Quantity  :",
    f"{product_warehouse_fulfillment['ordered_qty'].sum():,.0f}"
)

print(
    "Total Fulfilled Quantity:",
    f"{product_warehouse_fulfillment['fulfilled_qty'].sum():,.0f}"
)

print(
    "Total Backlog Quantity  :",
    f"{product_warehouse_fulfillment['backlog_qty'].sum():,.0f}"
)

print(
    "Total Backlog Value     :",
    f"{product_warehouse_fulfillment['backlog_value'].sum():,.2f}"
)

print(
    "Backlogged Lines        :",
    f"{product_warehouse_fulfillment['backlogged_lines'].sum():,}"
)

print(
    "\nProduct-Warehouse Fulfillment Shape:",
    product_warehouse_fulfillment.shape
)

PRODUCT-WAREHOUSE FULFILLMENT SUMMARY

Records                 : 2,985
Unique Products         : 500
Unique Warehouses       : 6
Total Order Lines       : 61,397
Total Ordered Quantity  : 501,755
Total Fulfilled Quantity: 440,576
Total Backlog Quantity  : 61,179
Total Backlog Value     : 257,349,919.69
Backlogged Lines        : 13,278

Product-Warehouse Fulfillment Shape: (2985, 10)


## Step 6 — Merge Inventory with Product-Warehouse Fulfillment

Combine the Product-Warehouse fulfillment summary with the current inventory position.

Inventory contains the complete Product × Warehouse structure of 3,000 records, while sales activity exists for 2,985 combinations.

Product-Warehouse combinations without historical sales activity will therefore be retained and assigned zero sales, fulfillment, and backlog activity.

This dataset provides the foundation for analyzing relationships between:

- Available inventory
- Inventory days on hand
- Safety stock
- Reorder position
- Demand
- Fulfillment rate
- Backlog quantity
- Backlog rate
- Backlog value

In [6]:
# Merge inventory with Product-Warehouse fulfillment performance

inventory_fulfillment = (
    inventory
    .merge(
        product_warehouse_fulfillment,
        on=["product_id", "warehouse_id"],
        how="left",
        validate="one_to_one"
    )
)


# Sales activity columns that should be zero when no historical sales exist

activity_columns = [
    "order_lines",
    "ordered_qty",
    "fulfilled_qty",
    "backlog_qty",
    "backlog_value",
    "backlogged_lines"
]


inventory_fulfillment[
    activity_columns
] = (
    inventory_fulfillment[
        activity_columns
    ]
    .fillna(0)
)


# Identify whether the Product-Warehouse combination had sales activity

inventory_fulfillment[
    "sales_activity_flag"
] = np.where(
    inventory_fulfillment["ordered_qty"] > 0,
    "Y",
    "N"
)


print("INVENTORY + FULFILLMENT RELATIONSHIP DATASET\n")

print(
    "Total Records               :",
    f"{len(inventory_fulfillment):,}"
)

print(
    "Unique Products             :",
    inventory_fulfillment["product_id"].nunique()
)

print(
    "Unique Warehouses           :",
    inventory_fulfillment["warehouse_id"].nunique()
)

print(
    "Records With Sales Activity :",
    (
        inventory_fulfillment["sales_activity_flag"]
        == "Y"
    ).sum()
)

print(
    "Records Without Sales       :",
    (
        inventory_fulfillment["sales_activity_flag"]
        == "N"
    ).sum()
)

print(
    "Total Ordered Quantity      :",
    f"{inventory_fulfillment['ordered_qty'].sum():,.0f}"
)

print(
    "Total Fulfilled Quantity    :",
    f"{inventory_fulfillment['fulfilled_qty'].sum():,.0f}"
)

print(
    "Total Backlog Quantity      :",
    f"{inventory_fulfillment['backlog_qty'].sum():,.0f}"
)

print(
    "Total Backlog Value         :",
    f"{inventory_fulfillment['backlog_value'].sum():,.2f}"
)

print(
    "\nDataset Shape:",
    inventory_fulfillment.shape
)

INVENTORY + FULFILLMENT RELATIONSHIP DATASET

Total Records               : 3,000
Unique Products             : 500
Unique Warehouses           : 6
Records With Sales Activity : 2985
Records Without Sales       : 15
Total Ordered Quantity      : 501,755
Total Fulfilled Quantity    : 440,576
Total Backlog Quantity      : 61,179
Total Backlog Value         : 257,349,919.69

Dataset Shape: (3000, 24)


## Step 7 — Inventory Availability vs Backlog Relationship

Analyze the relationship between current inventory availability and historical backlog performance at the Product-Warehouse level.

The analysis compares:

- Available Quantity vs Backlog Quantity
- Available Quantity vs Backlog Rate
- Inventory Days on Hand vs Backlog Rate
- Average Daily Demand vs Backlog Quantity

Only Product-Warehouse combinations with sales activity are included.

Pearson correlation is used as an initial measure of linear association.

> Note: Current inventory represents a point-in-time inventory position, while backlog reflects historical order activity. Therefore, the results should be interpreted as operational associations rather than evidence that current inventory levels directly caused historical backlog.

In [7]:
# Filter Product-Warehouse combinations with sales activity

inventory_backlog_analysis = (
    inventory_fulfillment[
        inventory_fulfillment["sales_activity_flag"] == "Y"
    ]
    .copy()
)


# Select relationship variables

relationship_columns = [
    "available_qty",
    "inventory_days_on_hand",
    "avg_daily_demand",
    "backlog_qty",
    "backlog_rate_pct"
]


inventory_backlog_corr = (
    inventory_backlog_analysis[
        relationship_columns
    ]
    .corr(method="pearson")
)


print("INVENTORY AVAILABILITY VS BACKLOG RELATIONSHIP\n")

print(
    "Records Analyzed:",
    f"{len(inventory_backlog_analysis):,}"
)


print(
    "\nAvailable Qty vs Backlog Qty      :",
    f"{inventory_backlog_corr.loc['available_qty', 'backlog_qty']:.4f}"
)

print(
    "Available Qty vs Backlog Rate     :",
    f"{inventory_backlog_corr.loc['available_qty', 'backlog_rate_pct']:.4f}"
)

print(
    "Inventory DOH vs Backlog Rate     :",
    f"{inventory_backlog_corr.loc['inventory_days_on_hand', 'backlog_rate_pct']:.4f}"
)

print(
    "Avg Daily Demand vs Backlog Qty   :",
    f"{inventory_backlog_corr.loc['avg_daily_demand', 'backlog_qty']:.4f}"
)


print("\nCORRELATION MATRIX\n")

print(
    inventory_backlog_corr.round(4)
)

INVENTORY AVAILABILITY VS BACKLOG RELATIONSHIP

Records Analyzed: 2,985

Available Qty vs Backlog Qty      : 0.5297
Available Qty vs Backlog Rate     : 0.2331
Inventory DOH vs Backlog Rate     : 0.0016
Avg Daily Demand vs Backlog Qty   : 0.7754

CORRELATION MATRIX

                        available_qty  inventory_days_on_hand  \
available_qty                  1.0000                  0.5684   
inventory_days_on_hand         0.5684                  1.0000   
avg_daily_demand               0.6635                 -0.0475   
backlog_qty                    0.5297                 -0.0254   
backlog_rate_pct               0.2331                  0.0016   

                        avg_daily_demand  backlog_qty  backlog_rate_pct  
available_qty                     0.6635       0.5297            0.2331  
inventory_days_on_hand           -0.0475      -0.0254            0.0016  
avg_daily_demand                  1.0000       0.7754            0.3300  
backlog_qty                       0.7754       

## Step 8 — Demand-Normalized Inventory Position vs Backlog

Raw inventory and backlog quantities are influenced by product demand volume.

To obtain a more meaningful operational comparison, Product-Warehouse records are grouped using demand-normalized inventory indicators:

- Inventory Days on Hand
- Reorder Required Status

For each group, backlog rate and quantity fill rate are compared.

This helps determine whether low inventory coverage or reorder pressure is associated with weaker fulfillment performance.

In [8]:
# Create inventory coverage bands

inventory_backlog_analysis[
    "inventory_coverage_band"
] = pd.cut(
    inventory_backlog_analysis["inventory_days_on_hand"],
    bins=[
        -np.inf,
        0,
        7,
        15,
        30,
        60,
        np.inf
    ],
    labels=[
        "0 Days",
        "1-7 Days",
        "8-15 Days",
        "16-30 Days",
        "31-60 Days",
        "61+ Days"
    ]
)


# Summarize backlog performance by inventory coverage

coverage_backlog_summary = (
    inventory_backlog_analysis
    .groupby(
        "inventory_coverage_band",
        observed=True
    )
    .agg(
        records=("product_id", "size"),
        ordered_qty=("ordered_qty", "sum"),
        fulfilled_qty=("fulfilled_qty", "sum"),
        backlog_qty=("backlog_qty", "sum"),
        backlog_value=("backlog_value", "sum")
    )
    .reset_index()
)


coverage_backlog_summary[
    "qty_fill_rate_pct"
] = (
    coverage_backlog_summary["fulfilled_qty"]
    / coverage_backlog_summary["ordered_qty"]
    * 100
)


coverage_backlog_summary[
    "backlog_rate_pct"
] = (
    coverage_backlog_summary["backlog_qty"]
    / coverage_backlog_summary["ordered_qty"]
    * 100
)


# Reorder status relationship

reorder_backlog_summary = (
    inventory_backlog_analysis
    .groupby(
        "reorder_required_flag",
        as_index=False
    )
    .agg(
        records=("product_id", "size"),
        ordered_qty=("ordered_qty", "sum"),
        fulfilled_qty=("fulfilled_qty", "sum"),
        backlog_qty=("backlog_qty", "sum"),
        backlog_value=("backlog_value", "sum")
    )
)


reorder_backlog_summary[
    "qty_fill_rate_pct"
] = (
    reorder_backlog_summary["fulfilled_qty"]
    / reorder_backlog_summary["ordered_qty"]
    * 100
)


reorder_backlog_summary[
    "backlog_rate_pct"
] = (
    reorder_backlog_summary["backlog_qty"]
    / reorder_backlog_summary["ordered_qty"]
    * 100
)


print("INVENTORY COVERAGE VS BACKLOG\n")

print(
    coverage_backlog_summary[
        [
            "inventory_coverage_band",
            "records",
            "ordered_qty",
            "backlog_qty",
            "qty_fill_rate_pct",
            "backlog_rate_pct",
            "backlog_value"
        ]
    ].round(2).to_string(index=False)
)


print("\nREORDER STATUS VS BACKLOG\n")

print(
    reorder_backlog_summary[
        [
            "reorder_required_flag",
            "records",
            "ordered_qty",
            "backlog_qty",
            "qty_fill_rate_pct",
            "backlog_rate_pct",
            "backlog_value"
        ]
    ].round(2).to_string(index=False)
)

INVENTORY COVERAGE VS BACKLOG

inventory_coverage_band  records  ordered_qty  backlog_qty  qty_fill_rate_pct  backlog_rate_pct  backlog_value
                 0 Days      240      37870.0       4678.0              87.65             12.35    17271257.24
               1-7 Days      174      32181.0       3784.0              88.24             11.76    16853323.39
              8-15 Days      326      57365.0       6849.0              88.06             11.94    26331917.56
             16-30 Days      688     115021.0      13737.0              88.06             11.94    56550050.18
             31-60 Days     1229     210874.0      26544.0              87.41             12.59   115491236.00
               61+ Days      328      48444.0       5587.0              88.47             11.53    24852135.32

REORDER STATUS VS BACKLOG

reorder_required_flag  records  ordered_qty  backlog_qty  qty_fill_rate_pct  backlog_rate_pct  backlog_value
                    N     1679     289477.0      36857.

## Step 9 — Stock Status vs Fulfillment & Backlog

Compare fulfillment and backlog performance across current inventory stock-status groups.

The analysis evaluates:

- Healthy
- Low Stock
- Stockout
- Overstock

For each stock status, we compare:

- Product-Warehouse records
- Ordered quantity
- Fulfilled quantity
- Backlog quantity
- Quantity fill rate
- Backlog rate
- Backlog value

Because stock status represents the current inventory position while order activity is historical, the results are interpreted as operational associations rather than causal effects.

In [9]:
# Analyze stock status vs fulfillment and backlog

stock_status_relationship = (
    inventory_backlog_analysis
    .groupby(
        "stock_status",
        as_index=False
    )
    .agg(
        records=("product_id", "size"),
        ordered_qty=("ordered_qty", "sum"),
        fulfilled_qty=("fulfilled_qty", "sum"),
        backlog_qty=("backlog_qty", "sum"),
        backlog_value=("backlog_value", "sum"),
        backlogged_lines=("backlogged_lines", "sum")
    )
)


# Calculate fulfillment KPIs

stock_status_relationship[
    "qty_fill_rate_pct"
] = (
    stock_status_relationship["fulfilled_qty"]
    / stock_status_relationship["ordered_qty"]
    * 100
)


stock_status_relationship[
    "backlog_rate_pct"
] = (
    stock_status_relationship["backlog_qty"]
    / stock_status_relationship["ordered_qty"]
    * 100
)


stock_status_relationship[
    "backlog_value_share_pct"
] = (
    stock_status_relationship["backlog_value"]
    / stock_status_relationship["backlog_value"].sum()
    * 100
)


# Sort by backlog rate

stock_status_relationship = (
    stock_status_relationship
    .sort_values(
        "backlog_rate_pct",
        ascending=False
    )
    .reset_index(drop=True)
)


print("STOCK STATUS VS FULFILLMENT & BACKLOG\n")


print(
    stock_status_relationship[
        [
            "stock_status",
            "records",
            "ordered_qty",
            "fulfilled_qty",
            "backlog_qty",
            "qty_fill_rate_pct",
            "backlog_rate_pct",
            "backlog_value",
            "backlog_value_share_pct"
        ]
    ]
    .round(2)
    .to_string(index=False)
)

STOCK STATUS VS FULFILLMENT & BACKLOG

stock_status  records  ordered_qty  fulfilled_qty  backlog_qty  qty_fill_rate_pct  backlog_rate_pct  backlog_value  backlog_value_share_pct
    Stockout      240      37870.0        33192.0       4678.0              87.65             12.35    17271257.24                     6.71
     Healthy     2064     344175.0       301984.0      42191.0              87.74             12.26   180344285.43                    70.08
   Overstock      476      81746.0        71842.0       9904.0              87.88             12.12    41393604.04                    16.08
   Low Stock      205      37964.0        33558.0       4406.0              88.39             11.61    18340772.98                     7.13


## Step 10 — Product Demand vs Backlog Relationship

Analyze whether higher-demand Product-Warehouse combinations experience greater backlog exposure.

Product-Warehouse records are divided into four demand quartiles based on average daily demand:

- Q1 — Lowest Demand
- Q2 — Low-Medium Demand
- Q3 — Medium-High Demand
- Q4 — Highest Demand

For each demand group, the analysis compares:

- Ordered quantity
- Fulfilled quantity
- Backlog quantity
- Backlog value
- Quantity fill rate
- Backlog rate

This helps distinguish whether backlog exposure is primarily associated with demand scale rather than inventory position alone.

In [10]:
# Create demand quartiles

inventory_backlog_analysis[
    "demand_quartile"
] = pd.qcut(
    inventory_backlog_analysis[
        "avg_daily_demand"
    ],
    q=4,
    labels=[
        "Q1 - Lowest Demand",
        "Q2 - Low-Medium Demand",
        "Q3 - Medium-High Demand",
        "Q4 - Highest Demand"
    ]
)


# Summarize fulfillment and backlog by demand quartile

demand_backlog_summary = (
    inventory_backlog_analysis
    .groupby(
        "demand_quartile",
        observed=True
    )
    .agg(
        records=("product_id", "size"),
        avg_daily_demand=(
            "avg_daily_demand",
            "mean"
        ),
        ordered_qty=("ordered_qty", "sum"),
        fulfilled_qty=("fulfilled_qty", "sum"),
        backlog_qty=("backlog_qty", "sum"),
        backlog_value=("backlog_value", "sum")
    )
    .reset_index()
)


demand_backlog_summary[
    "qty_fill_rate_pct"
] = (
    demand_backlog_summary["fulfilled_qty"]
    / demand_backlog_summary["ordered_qty"]
    * 100
)


demand_backlog_summary[
    "backlog_rate_pct"
] = (
    demand_backlog_summary["backlog_qty"]
    / demand_backlog_summary["ordered_qty"]
    * 100
)


demand_backlog_summary[
    "backlog_qty_share_pct"
] = (
    demand_backlog_summary["backlog_qty"]
    / demand_backlog_summary["backlog_qty"].sum()
    * 100
)


demand_backlog_summary[
    "backlog_value_share_pct"
] = (
    demand_backlog_summary["backlog_value"]
    / demand_backlog_summary["backlog_value"].sum()
    * 100
)


print("PRODUCT DEMAND VS BACKLOG\n")


print(
    demand_backlog_summary[
        [
            "demand_quartile",
            "records",
            "avg_daily_demand",
            "ordered_qty",
            "backlog_qty",
            "qty_fill_rate_pct",
            "backlog_rate_pct",
            "backlog_qty_share_pct",
            "backlog_value",
            "backlog_value_share_pct"
        ]
    ]
    .round(2)
    .to_string(index=False)
)

PRODUCT DEMAND VS BACKLOG

        demand_quartile  records  avg_daily_demand  ordered_qty  backlog_qty  qty_fill_rate_pct  backlog_rate_pct  backlog_qty_share_pct  backlog_value  backlog_value_share_pct
     Q1 - Lowest Demand      783              0.28      44281.0       3147.0              92.89              7.11                   5.14    13853538.00                     5.38
 Q2 - Low-Medium Demand      738              0.61      89276.0       6634.0              92.57              7.43                  10.84    29115030.65                    11.31
Q3 - Medium-High Demand      756              0.94     144855.0      11701.0              91.92              8.08                  19.13    48892965.98                    19.00
    Q4 - Highest Demand      708              1.57     223343.0      39697.0              82.23             17.77                  64.89   165488385.06                    64.30


## Step 11 — Product Sales Value vs Backlog Exposure

Analyze the relationship between product sales value and backlog exposure.

Sales order items are aggregated to the Product level to calculate:

- Total Sales Value
- Ordered Quantity
- Fulfilled Quantity
- Backlog Quantity
- Backlog Value
- Quantity Fill Rate
- Backlog Rate

Products are then divided into sales-value quartiles.

This analysis determines whether commercially important, high-revenue products also account for a disproportionate share of operational backlog exposure.

In [11]:
# Build product-level sales and backlog summary

product_sales_backlog = (
    sales_order_items
    .groupby(
        "product_id",
        as_index=False
    )
    .agg(
        order_lines=("order_item_id", "count"),
        sales_value=("net_order_value", "sum"),
        ordered_qty=("ordered_qty", "sum"),
        fulfilled_qty=("fulfilled_qty", "sum"),
        backlog_qty=("backlog_qty", "sum"),
        backlog_value=("backlog_value", "sum")
    )
)


# Calculate fulfillment KPIs

product_sales_backlog[
    "qty_fill_rate_pct"
] = (
    product_sales_backlog["fulfilled_qty"]
    / product_sales_backlog["ordered_qty"]
    * 100
)


product_sales_backlog[
    "backlog_rate_pct"
] = (
    product_sales_backlog["backlog_qty"]
    / product_sales_backlog["ordered_qty"]
    * 100
)


# Create sales-value quartiles

product_sales_backlog[
    "sales_value_quartile"
] = pd.qcut(
    product_sales_backlog["sales_value"],
    q=4,
    labels=[
        "Q1 - Lowest Sales Value",
        "Q2 - Low-Medium Sales Value",
        "Q3 - Medium-High Sales Value",
        "Q4 - Highest Sales Value"
    ]
)


# Summarize by sales-value quartile

sales_value_backlog_summary = (
    product_sales_backlog
    .groupby(
        "sales_value_quartile",
        observed=True
    )
    .agg(
        products=("product_id", "count"),
        sales_value=("sales_value", "sum"),
        ordered_qty=("ordered_qty", "sum"),
        fulfilled_qty=("fulfilled_qty", "sum"),
        backlog_qty=("backlog_qty", "sum"),
        backlog_value=("backlog_value", "sum")
    )
    .reset_index()
)


sales_value_backlog_summary[
    "qty_fill_rate_pct"
] = (
    sales_value_backlog_summary["fulfilled_qty"]
    / sales_value_backlog_summary["ordered_qty"]
    * 100
)


sales_value_backlog_summary[
    "backlog_rate_pct"
] = (
    sales_value_backlog_summary["backlog_qty"]
    / sales_value_backlog_summary["ordered_qty"]
    * 100
)


sales_value_backlog_summary[
    "sales_share_pct"
] = (
    sales_value_backlog_summary["sales_value"]
    / sales_value_backlog_summary["sales_value"].sum()
    * 100
)


sales_value_backlog_summary[
    "backlog_value_share_pct"
] = (
    sales_value_backlog_summary["backlog_value"]
    / sales_value_backlog_summary["backlog_value"].sum()
    * 100
)


# Product-level correlation

sales_backlog_correlation = (
    product_sales_backlog[
        [
            "sales_value",
            "backlog_qty",
            "backlog_value",
            "backlog_rate_pct"
        ]
    ]
    .corr()
)


print("PRODUCT SALES VALUE VS BACKLOG EXPOSURE\n")


print(
    sales_value_backlog_summary[
        [
            "sales_value_quartile",
            "products",
            "sales_value",
            "sales_share_pct",
            "backlog_qty",
            "backlog_rate_pct",
            "backlog_value",
            "backlog_value_share_pct"
        ]
    ]
    .round(2)
    .to_string(index=False)
)


print("\nPRODUCT-LEVEL CORRELATIONS\n")

print(
    "Sales Value vs Backlog Quantity :",
    f"{sales_backlog_correlation.loc['sales_value', 'backlog_qty']:.4f}"
)

print(
    "Sales Value vs Backlog Value    :",
    f"{sales_backlog_correlation.loc['sales_value', 'backlog_value']:.4f}"
)

print(
    "Sales Value vs Backlog Rate     :",
    f"{sales_backlog_correlation.loc['sales_value', 'backlog_rate_pct']:.4f}"
)

PRODUCT SALES VALUE VS BACKLOG EXPOSURE

        sales_value_quartile  products  sales_value  sales_share_pct  backlog_qty  backlog_rate_pct  backlog_value  backlog_value_share_pct
     Q1 - Lowest Sales Value       125 8.708129e+07             4.21         5975              9.14     6841088.62                     2.66
 Q2 - Low-Medium Sales Value       125 2.828718e+08            13.68        10025              9.69    24505453.35                     9.52
Q3 - Medium-High Sales Value       125 5.264504e+08            25.45        13567             10.84    50968520.69                    19.81
    Q4 - Highest Sales Value       125 1.172092e+09            56.66        31612             15.21   175034857.03                    68.01

PRODUCT-LEVEL CORRELATIONS

Sales Value vs Backlog Quantity : 0.6786
Sales Value vs Backlog Value    : 0.9218
Sales Value vs Backlog Rate     : 0.4126


## Step 12 — Fulfillment Rate vs Delivery Performance

Analyze the relationship between order fulfillment performance and downstream delivery outcomes.

Sales orders are connected with chronology-valid shipment records to compare:

- Order Fill Rate
- Delivery Delay
- On-Time Delivery
- OTIF

Orders are grouped into fulfillment-rate bands to determine whether incomplete fulfillment is associated with weaker delivery performance.

Only shipments with valid delivery chronology are included so that known shipment-date anomalies do not distort the results.

In [12]:
# Select chronology-valid shipment records

valid_shipments = (
    shipments[
        shipments["chronology_valid_flag"] == "Y"
    ][
        [
            "order_id",
            "delivery_delay_days_clean",
            "on_time_flag_clean",
            "otif_flag_clean"
        ]
    ]
    .copy()
)


# Merge order fulfillment with valid delivery performance

fulfillment_delivery = (
    sales_orders[
        [
            "order_id",
            "ordered_qty",
            "fulfilled_qty",
            "backlog_qty",
            "order_fill_rate_pct"
        ]
    ]
    .merge(
        valid_shipments,
        on="order_id",
        how="inner",
        validate="one_to_one"
    )
)


# Create fulfillment-rate bands

fulfillment_delivery[
    "fulfillment_rate_band"
] = pd.cut(
    fulfillment_delivery["order_fill_rate_pct"],
    bins=[
        -np.inf,
        60,
        80,
        90,
        99.999999,
        np.inf
    ],
    labels=[
        "<=60%",
        "60-80%",
        "80-90%",
        "90-<100%",
        "100%"
    ]
)


# Summarize delivery performance by fulfillment rate

fulfillment_delivery_summary = (
    fulfillment_delivery
    .groupby(
        "fulfillment_rate_band",
        observed=True
    )
    .agg(
        orders=("order_id", "count"),
        avg_fill_rate_pct=(
            "order_fill_rate_pct",
            "mean"
        ),
        avg_delivery_delay_days=(
            "delivery_delay_days_clean",
            "mean"
        ),
        on_time_rate_pct=(
            "on_time_flag_clean",
            lambda x: (x == "Y").mean() * 100
        ),
        otif_rate_pct=(
            "otif_flag_clean",
            lambda x: (x == "Y").mean() * 100
        )
    )
    .reset_index()
)


# Correlation between fulfillment rate and delivery delay

fill_delay_correlation = (
    fulfillment_delivery[
        [
            "order_fill_rate_pct",
            "delivery_delay_days_clean"
        ]
    ]
    .corr()
    .iloc[0, 1]
)


print("FULFILLMENT RATE VS DELIVERY PERFORMANCE\n")


print(
    fulfillment_delivery_summary
    .round(2)
    .to_string(index=False)
)


print(
    "\nOrder Fill Rate vs Delivery Delay Correlation :",
    f"{fill_delay_correlation:.4f}"
)

print(
    "Valid Orders Analyzed                         :",
    f"{len(fulfillment_delivery):,}"
)

FULFILLMENT RATE VS DELIVERY PERFORMANCE

fulfillment_rate_band  orders  avg_fill_rate_pct  avg_delivery_delay_days  on_time_rate_pct  otif_rate_pct
                <=60%    2154              44.68                     1.92             56.69           0.00
               60-80%    3019              71.81                     1.91             55.95           0.00
               80-90%    2464              85.58                     1.95             56.09           0.00
             90-<100%    1371              93.48                     1.87             57.18           0.00
                 100%   11323             100.00                     0.15             83.05          83.05

Order Fill Rate vs Delivery Delay Correlation : -0.1904
Valid Orders Analyzed                         : 20,331


## Step 13 — Backlog Severity vs Delivery Performance

Analyze the relationship between backlog severity and downstream delivery performance.

For each chronology-valid order, backlog is expressed as a percentage of ordered quantity.

Orders are then classified into:

- No Backlog
- Low Backlog
- Moderate Backlog
- High Backlog
- Critical Backlog

For each severity group, the analysis compares:

- Number of orders
- Average backlog percentage
- Average delivery delay
- On-Time Delivery rate
- OTIF rate

This helps determine whether increasing fulfillment shortfall is associated with worsening delivery performance.

In [13]:
# Calculate order-level backlog percentage

fulfillment_delivery[
    "backlog_pct"
] = np.where(
    fulfillment_delivery["ordered_qty"] > 0,
    (
        fulfillment_delivery["backlog_qty"]
        / fulfillment_delivery["ordered_qty"]
    ) * 100,
    0
)


# Create backlog severity bands

fulfillment_delivery[
    "backlog_severity"
] = pd.cut(
    fulfillment_delivery["backlog_pct"],
    bins=[
        -0.001,
        0,
        10,
        25,
        50,
        np.inf
    ],
    labels=[
        "No Backlog",
        "Low Backlog",
        "Moderate Backlog",
        "High Backlog",
        "Critical Backlog"
    ]
)


# Summarize delivery performance by backlog severity

backlog_delivery_summary = (
    fulfillment_delivery
    .groupby(
        "backlog_severity",
        observed=True
    )
    .agg(
        orders=("order_id", "count"),
        avg_backlog_pct=("backlog_pct", "mean"),
        avg_delivery_delay_days=(
            "delivery_delay_days_clean",
            "mean"
        ),
        on_time_rate_pct=(
            "on_time_flag_clean",
            lambda x: (x == "Y").mean() * 100
        ),
        otif_rate_pct=(
            "otif_flag_clean",
            lambda x: (x == "Y").mean() * 100
        )
    )
    .reset_index()
)


# Correlation between backlog percentage and delivery delay

backlog_delay_correlation = (
    fulfillment_delivery[
        [
            "backlog_pct",
            "delivery_delay_days_clean"
        ]
    ]
    .corr()
    .iloc[0, 1]
)


print("BACKLOG SEVERITY VS DELIVERY PERFORMANCE\n")


print(
    backlog_delivery_summary
    .round(2)
    .to_string(index=False)
)


print(
    "\nBacklog % vs Delivery Delay Correlation :",
    f"{backlog_delay_correlation:.4f}"
)

BACKLOG SEVERITY VS DELIVERY PERFORMANCE

backlog_severity  orders  avg_backlog_pct  avg_delivery_delay_days  on_time_rate_pct  otif_rate_pct
      No Backlog   11323             0.00                     0.15             83.05          83.05
     Low Backlog    1535             6.90                     1.88             57.00           0.00
Moderate Backlog    3527            17.47                     1.95             55.71           0.00
    High Backlog    2912            37.13                     1.96             56.04           0.00
Critical Backlog    1034            66.16                     1.74             58.41           0.00

Backlog % vs Delivery Delay Correlation : 0.1904


## Step 14 — Supplier Reliability vs Procurement Delay

Analyze whether supplier reliability is associated with actual procurement delivery performance.

Purchase orders are connected with supplier master data to compare:

- Reliability Factor
- Quality Rating
- Contract Lead Time
- Supplier Delay Days
- On-Time Procurement Performance

Only purchase orders with a recorded supplier delay are included.

This analysis helps determine whether supplier reliability indicators are meaningfully associated with actual procurement execution.

In [14]:
# Merge purchase orders with supplier attributes

supplier_procurement_relationship = (
    purchase_orders[
        [
            "po_id",
            "supplier_id",
            "supplier_delay_days"
        ]
    ]
    .merge(
        suppliers[
            [
                "supplier_id",
                "contract_lead_time_days",
                "quality_rating",
                "reliability_factor"
            ]
        ],
        on="supplier_id",
        how="left",
        validate="many_to_one"
    )
)


# Keep only POs with actual delivery performance

supplier_procurement_valid = (
    supplier_procurement_relationship[
        supplier_procurement_relationship[
            "supplier_delay_days"
        ].notna()
    ]
    .copy()
)


# Create procurement on-time flag

supplier_procurement_valid[
    "procurement_on_time_flag"
] = np.where(
    supplier_procurement_valid[
        "supplier_delay_days"
    ] <= 0,
    "Y",
    "N"
)


# Create supplier reliability bands

supplier_procurement_valid[
    "reliability_band"
] = pd.cut(
    supplier_procurement_valid[
        "reliability_factor"
    ],
    bins=[
        -np.inf,
        0.70,
        0.80,
        0.90,
        np.inf
    ],
    labels=[
        "Below 0.70",
        "0.70-0.79",
        "0.80-0.89",
        "0.90+"
    ],
    right=False
)


# Summarize procurement performance by reliability band

reliability_delay_summary = (
    supplier_procurement_valid
    .groupby(
        "reliability_band",
        observed=True
    )
    .agg(
        purchase_orders=("po_id", "count"),
        suppliers=("supplier_id", "nunique"),
        avg_reliability_factor=(
            "reliability_factor",
            "mean"
        ),
        avg_supplier_delay_days=(
            "supplier_delay_days",
            "mean"
        ),
        on_time_rate_pct=(
            "procurement_on_time_flag",
            lambda x: (x == "Y").mean() * 100
        )
    )
    .reset_index()
)


# Correlations

supplier_relationship_corr = (
    supplier_procurement_valid[
        [
            "reliability_factor",
            "quality_rating",
            "contract_lead_time_days",
            "supplier_delay_days"
        ]
    ]
    .corr()
)


print("SUPPLIER RELIABILITY VS PROCUREMENT DELAY\n")


print(
    reliability_delay_summary
    .round(2)
    .to_string(index=False)
)


print("\nSUPPLIER RELATIONSHIP CORRELATIONS\n")


print(
    "Reliability Factor vs Supplier Delay :",
    f"{supplier_relationship_corr.loc['reliability_factor', 'supplier_delay_days']:.4f}"
)

print(
    "Quality Rating vs Supplier Delay     :",
    f"{supplier_relationship_corr.loc['quality_rating', 'supplier_delay_days']:.4f}"
)

print(
    "Contract Lead Time vs Supplier Delay :",
    f"{supplier_relationship_corr.loc['contract_lead_time_days', 'supplier_delay_days']:.4f}"
)

print(
    "\nValid Purchase Orders Analyzed       :",
    f"{len(supplier_procurement_valid):,}"
)

SUPPLIER RELIABILITY VS PROCUREMENT DELAY

reliability_band  purchase_orders  suppliers  avg_reliability_factor  avg_supplier_delay_days  on_time_rate_pct
      Below 0.70              966         10                    0.64                     2.92             49.17
       0.80-0.89             3806         46                    0.85                     1.03             63.53
           0.90+             3698         44                    0.94                     0.09             71.44

SUPPLIER RELATIONSHIP CORRELATIONS

Reliability Factor vs Supplier Delay : -0.2320
Quality Rating vs Supplier Delay     : 0.0005
Contract Lead Time vs Supplier Delay : 0.0119

Valid Purchase Orders Analyzed       : 8,470


## Step 15 — Supplier Delay vs Inventory Risk

Analyze whether Product-Warehouse combinations exposed to greater supplier delays also show greater inventory risk.

Purchase-order performance is aggregated to the Product + Warehouse level and connected with current inventory indicators.

The analysis compares supplier delay exposure with:

- Available Quantity
- Inventory Days on Hand
- Reorder Requirement
- Stock Status

Supplier delay groups are then compared to determine whether poor procurement performance is associated with stronger replenishment pressure or weaker inventory availability.

Because current inventory represents a point-in-time position and procurement records cover historical activity, results are interpreted as associations rather than direct causal effects.

In [15]:
# Aggregate procurement performance to Product-Warehouse level

product_warehouse_procurement = (
    purchase_orders
    .groupby(
        ["product_id", "warehouse_id"],
        as_index=False
    )
    .agg(
        purchase_orders=("po_id", "count"),
        avg_supplier_delay_days=(
            "supplier_delay_days",
            "mean"
        ),
        max_supplier_delay_days=(
            "supplier_delay_days",
            "max"
        ),
        ordered_po_qty=("ordered_qty", "sum"),
        received_po_qty=("received_qty", "sum")
    )
)


# Merge procurement performance with inventory

supplier_inventory_relationship = (
    inventory
    .merge(
        product_warehouse_procurement,
        on=["product_id", "warehouse_id"],
        how="left",
        validate="one_to_one"
    )
)


# Keep Product-Warehouse records with measurable supplier delivery performance

supplier_inventory_valid = (
    supplier_inventory_relationship[
        supplier_inventory_relationship[
            "avg_supplier_delay_days"
        ].notna()
    ]
    .copy()
)


# Create supplier delay bands

supplier_inventory_valid[
    "supplier_delay_band"
] = pd.cut(
    supplier_inventory_valid[
        "avg_supplier_delay_days"
    ],
    bins=[
        -np.inf,
        0,
        2,
        5,
        np.inf
    ],
    labels=[
        "On-Time / Early",
        "0-2 Days Avg Delay",
        "2-5 Days Avg Delay",
        "5+ Days Avg Delay"
    ],
    include_lowest=True
)


# Summarize inventory risk by supplier delay band

supplier_delay_inventory_summary = (
    supplier_inventory_valid
    .groupby(
        "supplier_delay_band",
        observed=True
    )
    .agg(
        records=("product_id", "size"),
        avg_available_qty=(
            "available_qty",
            "mean"
        ),
        avg_inventory_doh=(
            "inventory_days_on_hand",
            "mean"
        ),
        reorder_required_rate_pct=(
            "reorder_required_flag",
            lambda x: (x == "Y").mean() * 100
        ),
        stockout_rate_pct=(
            "stock_status",
            lambda x: (x == "Stockout").mean() * 100
        )
    )
    .reset_index()
)


# Correlations

supplier_inventory_corr = (
    supplier_inventory_valid[
        [
            "avg_supplier_delay_days",
            "available_qty",
            "inventory_days_on_hand"
        ]
    ]
    .corr()
)


print("SUPPLIER DELAY VS INVENTORY RISK\n")


print(
    supplier_delay_inventory_summary
    .round(2)
    .to_string(index=False)
)


print("\nRELATIONSHIP CORRELATIONS\n")


print(
    "Avg Supplier Delay vs Available Qty :",
    f"{supplier_inventory_corr.loc['avg_supplier_delay_days', 'available_qty']:.4f}"
)

print(
    "Avg Supplier Delay vs Inventory DOH :",
    f"{supplier_inventory_corr.loc['avg_supplier_delay_days', 'inventory_days_on_hand']:.4f}"
)


print(
    "\nProduct-Warehouse Records Analyzed   :",
    f"{len(supplier_inventory_valid):,}"
)

SUPPLIER DELAY VS INVENTORY RISK

supplier_delay_band  records  avg_available_qty  avg_inventory_doh  reorder_required_rate_pct  stockout_rate_pct
    On-Time / Early     1466              27.58              32.10                      43.32               7.44
 0-2 Days Avg Delay      559              30.26              31.06                      44.19               8.23
 2-5 Days Avg Delay      301              32.82              31.42                      41.53               8.64
  5+ Days Avg Delay      196              28.51              34.88                      36.73               6.12

RELATIONSHIP CORRELATIONS

Avg Supplier Delay vs Available Qty : 0.0234
Avg Supplier Delay vs Inventory DOH : 0.0368

Product-Warehouse Records Analyzed   : 2,522


## Step 16 — Procurement Availability vs Replenishment Pressure

Analyze the relationship between incoming procurement quantities and current inventory replenishment pressure.

For each Product-Warehouse combination, purchase-order quantities are aggregated to calculate:

- Total PO Ordered Quantity
- Total PO Received Quantity
- Outstanding PO Quantity

These procurement indicators are connected with current inventory data to compare:

- Reorder-required records
- Available inventory
- Inventory days on hand
- Outstanding inbound quantity

This analysis helps determine whether Product-Warehouse combinations currently requiring replenishment already have inbound procurement coverage.

In [16]:
# Calculate Product-Warehouse procurement availability

procurement_availability = (
    purchase_orders
    .groupby(
        ["product_id", "warehouse_id"],
        as_index=False
    )
    .agg(
        po_count=("po_id", "count"),
        po_ordered_qty=("ordered_qty", "sum"),
        po_received_qty=("received_qty", "sum")
    )
)


# Calculate outstanding PO quantity

procurement_availability[
    "outstanding_po_qty"
] = (
    procurement_availability["po_ordered_qty"]
    - procurement_availability["po_received_qty"]
)


# Merge with current inventory

inventory_procurement = (
    inventory
    .merge(
        procurement_availability,
        on=["product_id", "warehouse_id"],
        how="left",
        validate="one_to_one"
    )
)


# Product-Warehouse combinations without POs receive zero procurement activity

procurement_columns = [
    "po_count",
    "po_ordered_qty",
    "po_received_qty",
    "outstanding_po_qty"
]

inventory_procurement[
    procurement_columns
] = (
    inventory_procurement[
        procurement_columns
    ]
    .fillna(0)
)


# Identify whether inbound quantity exists

inventory_procurement[
    "inbound_po_flag"
] = np.where(
    inventory_procurement["outstanding_po_qty"] > 0,
    "Y",
    "N"
)


# Summarize by reorder requirement

reorder_procurement_summary = (
    inventory_procurement
    .groupby(
        "reorder_required_flag",
        as_index=False
    )
    .agg(
        records=("product_id", "size"),
        avg_available_qty=("available_qty", "mean"),
        avg_inventory_doh=(
            "inventory_days_on_hand",
            "mean"
        ),
        records_with_inbound=(
            "inbound_po_flag",
            lambda x: (x == "Y").sum()
        ),
        total_outstanding_po_qty=(
            "outstanding_po_qty",
            "sum"
        )
    )
)


reorder_procurement_summary[
    "inbound_coverage_record_pct"
] = (
    reorder_procurement_summary["records_with_inbound"]
    / reorder_procurement_summary["records"]
    * 100
)


print("PROCUREMENT AVAILABILITY VS REPLENISHMENT PRESSURE\n")


print(
    reorder_procurement_summary
    .round(2)
    .to_string(index=False)
)


print(
    "\nTotal Product-Warehouse Records :",
    f"{len(inventory_procurement):,}"
)

print(
    "Records With Inbound PO Qty     :",
    f"{(inventory_procurement['inbound_po_flag'] == 'Y').sum():,}"
)

print(
    "Total Outstanding PO Quantity   :",
    f"{inventory_procurement['outstanding_po_qty'].sum():,.0f}"
)

PROCUREMENT AVAILABILITY VS REPLENISHMENT PRESSURE

reorder_required_flag  records  avg_available_qty  avg_inventory_doh  records_with_inbound  total_outstanding_po_qty  inbound_coverage_record_pct
                    N     1689              38.66              46.99                  1445                   76020.0                        85.55
                    Y     1311               9.86              13.10                  1080                   56806.0                        82.38

Total Product-Warehouse Records : 3,000
Records With Inbound PO Qty     : 2,525
Total Outstanding PO Quantity   : 132,826


## Step 17 — Warehouse-Level Cross-Functional Relationship

Build a warehouse-level relationship view connecting multiple operational functions.

For each warehouse, the analysis combines:

- Inventory availability
- Reorder pressure
- Order fulfillment
- Backlog
- On-Time Delivery
- OTIF
- Supplier procurement delay

This provides a cross-functional view of warehouse performance and helps identify whether operational weaknesses occur together across inventory, fulfillment, delivery, and procurement.

In [17]:
# Inventory metrics by warehouse

warehouse_inventory = (
    inventory
    .groupby(
        "warehouse_id",
        as_index=False
    )
    .agg(
        available_qty=("available_qty", "sum"),
        inventory_value=("inventory_value", "sum"),
        avg_inventory_doh=(
            "inventory_days_on_hand",
            "mean"
        ),
        reorder_required_rate_pct=(
            "reorder_required_flag",
            lambda x: (x == "Y").mean() * 100
        )
    )
)


# Fulfillment metrics by warehouse

warehouse_fulfillment = (
    sales_orders
    .groupby(
        "warehouse_id",
        as_index=False
    )
    .agg(
        orders=("order_id", "count"),
        ordered_qty=("ordered_qty", "sum"),
        fulfilled_qty=("fulfilled_qty", "sum"),
        backlog_qty=("backlog_qty", "sum"),
        backlog_value=("backlog_value", "sum")
    )
)


warehouse_fulfillment[
    "qty_fill_rate_pct"
] = (
    warehouse_fulfillment["fulfilled_qty"]
    / warehouse_fulfillment["ordered_qty"]
    * 100
)


warehouse_fulfillment[
    "backlog_rate_pct"
] = (
    warehouse_fulfillment["backlog_qty"]
    / warehouse_fulfillment["ordered_qty"]
    * 100
)


# Valid delivery metrics by warehouse

warehouse_delivery = (
    shipments[
        shipments["chronology_valid_flag"] == "Y"
    ]
    .groupby(
        "warehouse_id",
        as_index=False
    )
    .agg(
        valid_shipments=("shipment_id", "count"),
        avg_delivery_delay_days=(
            "delivery_delay_days_clean",
            "mean"
        ),
        on_time_rate_pct=(
            "on_time_flag_clean",
            lambda x: (x == "Y").mean() * 100
        ),
        otif_rate_pct=(
            "otif_flag_clean",
            lambda x: (x == "Y").mean() * 100
        )
    )
)


# Procurement metrics by warehouse

warehouse_procurement = (
    purchase_orders[
        purchase_orders[
            "supplier_delay_days"
        ].notna()
    ]
    .groupby(
        "warehouse_id",
        as_index=False
    )
    .agg(
        purchase_orders=("po_id", "count"),
        avg_supplier_delay_days=(
            "supplier_delay_days",
            "mean"
        ),
        procurement_on_time_rate_pct=(
            "supplier_delay_days",
            lambda x: (x <= 0).mean() * 100
        )
    )
)


# Combine warehouse metrics

warehouse_relationship = (
    warehouses[
        [
            "warehouse_id",
            "warehouse_name",
            "region"
        ]
    ]
    .merge(
        warehouse_inventory,
        on="warehouse_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        warehouse_fulfillment,
        on="warehouse_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        warehouse_delivery,
        on="warehouse_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        warehouse_procurement,
        on="warehouse_id",
        how="left",
        validate="one_to_one"
    )
)


print("WAREHOUSE-LEVEL CROSS-FUNCTIONAL RELATIONSHIP\n")


display_columns = [
    "warehouse_id",
    "warehouse_name",
    "reorder_required_rate_pct",
    "qty_fill_rate_pct",
    "backlog_rate_pct",
    "on_time_rate_pct",
    "otif_rate_pct",
    "avg_supplier_delay_days",
    "procurement_on_time_rate_pct"
]


print(
    warehouse_relationship[
        display_columns
    ]
    .round(2)
    .to_string(index=False)
)

WAREHOUSE-LEVEL CROSS-FUNCTIONAL RELATIONSHIP

warehouse_id    warehouse_name  reorder_required_rate_pct  qty_fill_rate_pct  backlog_rate_pct  on_time_rate_pct  otif_rate_pct  avg_supplier_delay_days  procurement_on_time_rate_pct
        WH01 Bengaluru Central                       46.8              87.96             12.04             71.42          45.91                     0.81                         63.97
        WH02       Mumbai West                       42.4              88.27             11.73             72.22          46.16                     0.90                         66.71
        WH03         Delhi NCR                       41.2              88.19             11.81             71.38          47.12                     0.76                         65.31
        WH04   Hyderabad South                       41.2              87.77             12.23             70.73          46.73                     0.75                         64.83
        WH05       Chennai Hub        

## Step 18 — Category-Level Cross-Functional Relationship

Build a category-level relationship view connecting commercial and operational performance.

For each product category, the analysis combines:

- Sales value
- Backlog value
- Quantity fill rate
- Backlog rate
- Current inventory value
- Reorder pressure
- Inventory days on hand

This helps identify categories where high commercial importance overlaps with inventory and fulfillment risk.

In [18]:
# Add category to inventory

category_inventory_base = (
    inventory
    .merge(
        products[
            [
                "product_id",
                "category"
            ]
        ],
        on="product_id",
        how="left",
        validate="many_to_one"
    )
)


# Category inventory metrics

category_inventory = (
    category_inventory_base
    .groupby(
        "category",
        as_index=False
    )
    .agg(
        inventory_value=(
            "inventory_value",
            "sum"
        ),
        available_qty=(
            "available_qty",
            "sum"
        ),
        avg_inventory_doh=(
            "inventory_days_on_hand",
            "mean"
        ),
        reorder_required_rate_pct=(
            "reorder_required_flag",
            lambda x: (x == "Y").mean() * 100
        )
    )
)


# Add category to sales-order items

category_sales_base = (
    sales_order_items
    .merge(
        products[
            [
                "product_id",
                "category"
            ]
        ],
        on="product_id",
        how="left",
        validate="many_to_one"
    )
)


# Category sales / fulfillment metrics

category_fulfillment = (
    category_sales_base
    .groupby(
        "category",
        as_index=False
    )
    .agg(
        order_lines=("order_item_id", "count"),
        sales_value=("net_order_value", "sum"),
        ordered_qty=("ordered_qty", "sum"),
        fulfilled_qty=("fulfilled_qty", "sum"),
        backlog_qty=("backlog_qty", "sum"),
        backlog_value=("backlog_value", "sum")
    )
)


category_fulfillment[
    "qty_fill_rate_pct"
] = (
    category_fulfillment["fulfilled_qty"]
    / category_fulfillment["ordered_qty"]
    * 100
)


category_fulfillment[
    "backlog_rate_pct"
] = (
    category_fulfillment["backlog_qty"]
    / category_fulfillment["ordered_qty"]
    * 100
)


# Combine category metrics

category_relationship = (
    category_fulfillment
    .merge(
        category_inventory,
        on="category",
        how="left",
        validate="one_to_one"
    )
)


# Calculate commercial exposure

category_relationship[
    "sales_share_pct"
] = (
    category_relationship["sales_value"]
    / category_relationship["sales_value"].sum()
    * 100
)


category_relationship[
    "backlog_value_share_pct"
] = (
    category_relationship["backlog_value"]
    / category_relationship["backlog_value"].sum()
    * 100
)


category_relationship = (
    category_relationship
    .sort_values(
        "backlog_value",
        ascending=False
    )
    .reset_index(drop=True)
)


print("CATEGORY-LEVEL CROSS-FUNCTIONAL RELATIONSHIP\n")


print(
    category_relationship[
        [
            "category",
            "sales_value",
            "sales_share_pct",
            "backlog_value",
            "backlog_value_share_pct",
            "qty_fill_rate_pct",
            "backlog_rate_pct",
            "inventory_value",
            "avg_inventory_doh",
            "reorder_required_rate_pct"
        ]
    ]
    .round(2)
    .to_string(index=False)
)

CATEGORY-LEVEL CROSS-FUNCTIONAL RELATIONSHIP

              category  sales_value  sales_share_pct  backlog_value  backlog_value_share_pct  qty_fill_rate_pct  backlog_rate_pct  inventory_value  avg_inventory_doh  reorder_required_rate_pct
           Electronics 594394618.00            28.74    72419213.94                    28.14              87.91             12.09      79180029.99              31.78                      42.76
       Office Supplies 392745467.90            18.99    49295528.24                    19.16              88.02             11.98      55462605.80              31.83                      44.33
Health & Personal Care 367898426.65            17.79    46802484.94                    18.19              87.49             12.51      50175615.88              31.72                      47.05
            Automotive 334110488.16            16.15    45572410.77                    17.71              86.63             13.37      47216372.95              31.83                 

## Step 19 — Overall Operational KPI Correlation Matrix

Build a consolidated correlation matrix across the major Product-Warehouse operational indicators.

The matrix evaluates relationships among:

- Average Daily Demand
- Available Inventory
- Inventory Days on Hand
- Inventory Value
- Ordered Quantity
- Fulfilled Quantity
- Backlog Quantity
- Backlog Value
- Quantity Fill Rate
- Backlog Rate

This provides a single statistical view of the strongest linear relationships across demand, inventory, fulfillment, and backlog performance.

Correlation measures association only and should not be interpreted as proof of causation.

In [19]:
# Select Product-Warehouse operational KPIs

correlation_columns = [
    "avg_daily_demand",
    "available_qty",
    "inventory_days_on_hand",
    "inventory_value",
    "ordered_qty",
    "fulfilled_qty",
    "backlog_qty",
    "backlog_value",
    "qty_fill_rate_pct",
    "backlog_rate_pct"
]


operational_correlation_matrix = (
    inventory_backlog_analysis[
        correlation_columns
    ]
    .corr(method="pearson")
)


print("OVERALL OPERATIONAL KPI CORRELATION MATRIX\n")


print(
    operational_correlation_matrix
    .round(3)
    .to_string()
)


# Extract strongest unique relationships

correlation_pairs = []


for i in range(len(correlation_columns)):

    for j in range(i + 1, len(correlation_columns)):

        variable_1 = correlation_columns[i]
        variable_2 = correlation_columns[j]

        correlation_value = (
            operational_correlation_matrix
            .loc[
                variable_1,
                variable_2
            ]
        )

        correlation_pairs.append(
            [
                variable_1,
                variable_2,
                correlation_value,
                abs(correlation_value)
            ]
        )


strongest_relationships = pd.DataFrame(
    correlation_pairs,
    columns=[
        "variable_1",
        "variable_2",
        "correlation",
        "absolute_correlation"
    ]
)


strongest_relationships = (
    strongest_relationships
    .sort_values(
        "absolute_correlation",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\nTOP 10 STRONGEST KPI RELATIONSHIPS\n")


print(
    strongest_relationships[
        [
            "variable_1",
            "variable_2",
            "correlation"
        ]
    ]
    .head(10)
    .round(4)
    .to_string(index=False)
)

OVERALL OPERATIONAL KPI CORRELATION MATRIX

                        avg_daily_demand  available_qty  inventory_days_on_hand  inventory_value  ordered_qty  fulfilled_qty  backlog_qty  backlog_value  qty_fill_rate_pct  backlog_rate_pct
avg_daily_demand                   1.000          0.664                  -0.048            0.542        0.873          0.844        0.775          0.628             -0.330             0.330
available_qty                      0.664          1.000                   0.568            0.794        0.573          0.548        0.530          0.478             -0.233             0.233
inventory_days_on_hand            -0.048          0.568                   1.000            0.420       -0.049         -0.052       -0.025         -0.003             -0.002             0.002
inventory_value                    0.542          0.794                   0.420            1.000        0.467          0.440        0.460          0.659             -0.198             0.198
ordere

## Step 20 — Identify Meaningful Independent Relationships

The overall correlation matrix contains both meaningful operational relationships and relationships that are mechanically created by KPI formulas.

For example:

- Quantity Fill Rate and Backlog Rate are exact complements.
- Ordered Quantity and Fulfilled Quantity are directly connected through the fulfillment equation.
- Backlog Quantity and Backlog Value are naturally related because backlog value is derived from unfulfilled quantity and price.

Therefore, these relationships should not be treated as independent business discoveries.

This step focuses on selected relationships that connect different operational concepts and are more useful for business interpretation.

In [20]:
# Define selected cross-functional relationships

meaningful_relationships = pd.DataFrame(
    [
        [
            "Demand vs Backlog Quantity",
            operational_correlation_matrix.loc[
                "avg_daily_demand",
                "backlog_qty"
            ]
        ],
        [
            "Demand vs Backlog Rate",
            operational_correlation_matrix.loc[
                "avg_daily_demand",
                "backlog_rate_pct"
            ]
        ],
        [
            "Available Qty vs Backlog Rate",
            operational_correlation_matrix.loc[
                "available_qty",
                "backlog_rate_pct"
            ]
        ],
        [
            "Inventory DOH vs Backlog Rate",
            operational_correlation_matrix.loc[
                "inventory_days_on_hand",
                "backlog_rate_pct"
            ]
        ],
        [
            "Inventory Value vs Backlog Value",
            operational_correlation_matrix.loc[
                "inventory_value",
                "backlog_value"
            ]
        ],
        [
            "Order Fill Rate vs Delivery Delay",
            fill_delay_correlation
        ],
        [
            "Backlog % vs Delivery Delay",
            backlog_delay_correlation
        ],
        [
            "Supplier Reliability vs Supplier Delay",
            supplier_relationship_corr.loc[
                "reliability_factor",
                "supplier_delay_days"
            ]
        ],
        [
            "Supplier Delay vs Available Qty",
            supplier_inventory_corr.loc[
                "avg_supplier_delay_days",
                "available_qty"
            ]
        ],
        [
            "Supplier Delay vs Inventory DOH",
            supplier_inventory_corr.loc[
                "avg_supplier_delay_days",
                "inventory_days_on_hand"
            ]
        ]
    ],
    columns=[
        "relationship",
        "correlation"
    ]
)


# Add absolute correlation

meaningful_relationships[
    "absolute_correlation"
] = (
    meaningful_relationships[
        "correlation"
    ].abs()
)


# Classify relationship strength

meaningful_relationships[
    "relationship_strength"
] = pd.cut(
    meaningful_relationships[
        "absolute_correlation"
    ],
    bins=[
        -np.inf,
        0.10,
        0.30,
        0.50,
        0.70,
        np.inf
    ],
    labels=[
        "Negligible",
        "Weak",
        "Moderate",
        "Strong",
        "Very Strong"
    ]
)


meaningful_relationships = (
    meaningful_relationships
    .sort_values(
        "absolute_correlation",
        ascending=False
    )
    .reset_index(drop=True)
)


print("MEANINGFUL CROSS-FUNCTIONAL RELATIONSHIPS\n")


print(
    meaningful_relationships[
        [
            "relationship",
            "correlation",
            "relationship_strength"
        ]
    ]
    .round(
        {
            "correlation": 4
        }
    )
    .to_string(index=False)
)

MEANINGFUL CROSS-FUNCTIONAL RELATIONSHIPS

                          relationship  correlation relationship_strength
            Demand vs Backlog Quantity       0.7754           Very Strong
      Inventory Value vs Backlog Value       0.6592                Strong
                Demand vs Backlog Rate       0.3300              Moderate
         Available Qty vs Backlog Rate       0.2331                  Weak
Supplier Reliability vs Supplier Delay      -0.2320                  Weak
           Backlog % vs Delivery Delay       0.1904                  Weak
     Order Fill Rate vs Delivery Delay      -0.1904                  Weak
       Supplier Delay vs Inventory DOH       0.0368            Negligible
       Supplier Delay vs Available Qty       0.0234            Negligible
         Inventory DOH vs Backlog Rate       0.0016            Negligible


## Step 21 — Relationship Analysis Key Findings Summary

Consolidate the most important cross-functional findings identified throughout the relationship analysis.

The purpose of this summary is to distinguish:

- Strong operational relationships
- Threshold effects
- Weak or negligible relationships
- Commercial risk concentrations
- Cross-functional operational hotspots

These findings will form the evidence base for the final business interpretation.

In [21]:
# Build consolidated relationship findings

relationship_findings = pd.DataFrame(
    [
        [
            "Demand → Backlog",
            "Very Strong",
            "Avg Daily Demand vs Backlog Qty correlation = 0.7754",
            "High-demand combinations carry substantially greater backlog exposure"
        ],

        [
            "Highest Demand Quartile",
            "High Risk Concentration",
            "64.89% of backlog quantity and 64.30% of backlog value",
            "Demand concentration is a major operational backlog risk"
        ],

        [
            "Sales Value → Backlog Value",
            "Very Strong",
            "Product-level correlation = 0.9218",
            "Commercially important products carry disproportionate backlog exposure"
        ],

        [
            "Highest Sales-Value Quartile",
            "Commercial Risk",
            "56.66% of sales but 68.01% of backlog value",
            "Backlog exposure is concentrated in high-value products"
        ],

        [
            "Inventory DOH → Backlog Rate",
            "Negligible",
            "Correlation = 0.0016",
            "Current inventory coverage alone does not explain historical backlog"
        ],

        [
            "Stock Status → Backlog",
            "Weak Differentiation",
            "Backlog rates range approximately 11.61%–12.35%",
            "Current stock-status groups show limited separation in historical backlog"
        ],

        [
            "Complete Fulfillment → Delivery",
            "Strong Threshold Effect",
            "100% fulfilled orders achieve 83.05% On-Time vs ~56–57% for partial fulfillment",
            "Complete fulfillment is strongly associated with better delivery service"
        ],

        [
            "Backlog Presence → Delivery",
            "Strong Threshold Effect",
            "No-backlog orders achieve 83.05% On-Time vs ~56–58% for backlogged orders",
            "Presence of backlog matters more than backlog severity"
        ],

        [
            "Supplier Reliability → Delay",
            "Weak Inverse Relationship",
            "Correlation = -0.2320",
            "More reliable suppliers tend to experience lower procurement delay"
        ],

        [
            "High Supplier Reliability",
            "Better Procurement",
            "0.90+ suppliers achieve 71.44% On-Time with 0.09 average delay days",
            "Reliability factor provides useful supplier-performance differentiation"
        ],

        [
            "Supplier Delay → Current Inventory",
            "Negligible",
            "Delay vs Available Qty = 0.0234; Delay vs DOH = 0.0368",
            "Historical supplier delay does not explain current inventory position"
        ],

        [
            "Replenishment → Inbound Supply",
            "High Coverage",
            "82.38% of reorder-required records already have inbound PO quantity",
            "Procurement is responding to most current replenishment signals"
        ],

        [
            "Pune DC",
            "Warehouse Risk",
            "48.20% reorder rate, 12.88% backlog rate, 70.54% On-Time, 45.39% OTIF",
            "Multiple operational weaknesses overlap at Pune DC"
        ],

        [
            "Automotive",
            "Category Risk",
            "86.63% fill rate and 13.37% backlog rate",
            "Automotive shows the weakest category-level fulfillment performance"
        ],

        [
            "Electronics",
            "Commercial Exposure",
            "28.74% of sales and 28.14% of backlog value",
            "Electronics carries the largest absolute commercial backlog exposure"
        ]
    ],

    columns=[
        "relationship_area",
        "finding_type",
        "evidence",
        "business_interpretation"
    ]
)


print("RELATIONSHIP ANALYSIS — KEY FINDINGS SUMMARY\n")


print(
    relationship_findings
    .to_string(index=False)
)


print(
    "\nTotal Consolidated Findings:",
    len(relationship_findings)
)

RELATIONSHIP ANALYSIS — KEY FINDINGS SUMMARY

                 relationship_area              finding_type                                                                        evidence                                                   business_interpretation
                  Demand → Backlog               Very Strong                            Avg Daily Demand vs Backlog Qty correlation = 0.7754     High-demand combinations carry substantially greater backlog exposure
           Highest Demand Quartile   High Risk Concentration                          64.89% of backlog quantity and 64.30% of backlog value                  Demand concentration is a major operational backlog risk
       Sales Value → Backlog Value               Very Strong                                              Product-level correlation = 0.9218   Commercially important products carry disproportionate backlog exposure
      Highest Sales-Value Quartile           Commercial Risk                                  

## Step 22 — Relationship Analysis Reconciliation

Perform a comprehensive reconciliation of the datasets and analytical tables created during the Relationship Analysis.

The objective is to verify that:

- Source datasets retain their expected record counts
- Product-Warehouse aggregation preserves sales quantities
- Inventory-Fulfillment integration retains the complete 3,000-record inventory structure
- Valid shipment counts reconcile with the known chronology-valid population
- Procurement records reconcile correctly
- Warehouse and category relationship tables contain the expected dimensions
- Consolidated relationship findings are complete

This ensures that the cross-functional relationships are based on internally consistent analytical data.

In [22]:
# Relationship Analysis Reconciliation

relationship_reconciliation_checks = {

    # -------------------------------------------------
    # Source datasets
    # -------------------------------------------------

    "Sales Order Count":
        len(sales_orders) == 25_000,

    "Sales Order Item Count":
        len(sales_order_items) == 61_397,

    "Inventory Record Count":
        len(inventory) == 3_000,

    "Inventory Transaction Count":
        len(inventory_transactions) == 110_000,

    "Shipment Record Count":
        len(shipments) == 24_798,

    "Purchase Order Count":
        len(purchase_orders) == 8_500,

    "Product Count":
        len(products) == 500,

    "Supplier Count":
        len(suppliers) == 100,

    "Warehouse Count":
        len(warehouses) == 6,


    # -------------------------------------------------
    # Product-Warehouse fulfillment reconciliation
    # -------------------------------------------------

    "Product-Warehouse Sales Records":
        len(product_warehouse_fulfillment) == 2_985,

    "PW Ordered Quantity":
        np.isclose(
            product_warehouse_fulfillment[
                "ordered_qty"
            ].sum(),
            501_755
        ),

    "PW Fulfilled Quantity":
        np.isclose(
            product_warehouse_fulfillment[
                "fulfilled_qty"
            ].sum(),
            440_576
        ),

    "PW Backlog Quantity":
        np.isclose(
            product_warehouse_fulfillment[
                "backlog_qty"
            ].sum(),
            61_179
        ),

    "PW Backlog Value":
        np.isclose(
            product_warehouse_fulfillment[
                "backlog_value"
            ].sum(),
            257_349_919.69,
            atol=0.01
        ),


    # -------------------------------------------------
    # Inventory + fulfillment relationship
    # -------------------------------------------------

    "Inventory-Fulfillment Record Count":
        len(inventory_fulfillment) == 3_000,

    "Records With Sales Activity":
        (
            inventory_fulfillment[
                "sales_activity_flag"
            ] == "Y"
        ).sum() == 2_985,

    "Records Without Sales Activity":
        (
            inventory_fulfillment[
                "sales_activity_flag"
            ] == "N"
        ).sum() == 15,

    "Inventory-Fulfillment Backlog Reconciles":
        np.isclose(
            inventory_fulfillment[
                "backlog_qty"
            ].sum(),
            61_179
        ),


    # -------------------------------------------------
    # Delivery relationship
    # -------------------------------------------------

    "Valid Delivery Order Count":
        len(fulfillment_delivery) == 20_331,

    "Valid Shipment Chronology Count":
        (
            shipments[
                "chronology_valid_flag"
            ] == "Y"
        ).sum() == 20_331,

    "Invalid Shipment Chronology Count":
        (
            shipments[
                "chronology_valid_flag"
            ] == "N"
        ).sum() == 4_467,


    # -------------------------------------------------
    # Procurement relationships
    # -------------------------------------------------

    "Valid Procurement Delivery Count":
        len(supplier_procurement_valid) == 8_470,

    "Supplier-Inventory Records Analyzed":
        len(supplier_inventory_valid) == 2_522,

    "Inventory-Procurement Record Count":
        len(inventory_procurement) == 3_000,

    "Records With Inbound PO Quantity":
        (
            inventory_procurement[
                "inbound_po_flag"
            ] == "Y"
        ).sum() == 2_525,

    "Outstanding PO Quantity":
        np.isclose(
            inventory_procurement[
                "outstanding_po_qty"
            ].sum(),
            132_826
        ),


    # -------------------------------------------------
    # Cross-functional dimensions
    # -------------------------------------------------

    "Warehouse Relationship Count":
        len(warehouse_relationship) == 6,

    "Category Relationship Count":
        len(category_relationship) == 5,

    "Relationship Findings Count":
        len(relationship_findings) == 15
}


print("RELATIONSHIP ANALYSIS RECONCILIATION\n")


for check, result in relationship_reconciliation_checks.items():

    print(
        f"{check:<45} : "
        f"{'PASS' if result else 'FAIL'}"
    )


relationship_reconciliation_passed = (
    all(
        relationship_reconciliation_checks.values()
    )
)


print(
    "\nRELATIONSHIP ANALYSIS RECONCILIATION STATUS :",
    "PASSED"
    if relationship_reconciliation_passed
    else "REVIEW REQUIRED"
)

RELATIONSHIP ANALYSIS RECONCILIATION

Sales Order Count                             : PASS
Sales Order Item Count                        : PASS
Inventory Record Count                        : PASS
Inventory Transaction Count                   : PASS
Shipment Record Count                         : PASS
Purchase Order Count                          : PASS
Product Count                                 : PASS
Supplier Count                                : PASS
Warehouse Count                               : PASS
Product-Warehouse Sales Records               : PASS
PW Ordered Quantity                           : PASS
PW Fulfilled Quantity                         : PASS
PW Backlog Quantity                           : PASS
PW Backlog Value                              : PASS
Inventory-Fulfillment Record Count            : PASS
Records With Sales Activity                   : PASS
Records Without Sales Activity                : PASS
Inventory-Fulfillment Backlog Reconciles      : PASS
Valid De

## Step 23 — Final Business Insights

### 1. Demand Intensity Is the Strongest Operational Driver of Backlog Exposure

Average Daily Demand has a **0.7754 correlation with Backlog Quantity**, representing the strongest meaningful cross-functional relationship identified at the Product-Warehouse level.

The highest-demand quartile alone accounts for:

- **64.89% of total backlog quantity**
- **64.30% of total backlog value**
- **17.77% backlog rate**
- Only **82.23% quantity fill rate**

In comparison, the lower three demand quartiles maintain fill rates above **91%**.

This indicates that backlog exposure is heavily concentrated in high-demand Product-Warehouse combinations.

---

### 2. Commercially Important Products Carry Disproportionate Backlog Risk

At the product level:

- Sales Value vs Backlog Quantity correlation: **0.6786**
- Sales Value vs Backlog Value correlation: **0.9218**
- Sales Value vs Backlog Rate correlation: **0.4126**

The highest-sales-value quartile generates:

- **56.66% of total sales**
- **68.01% of total backlog value**
- **15.21% backlog rate**

Therefore, backlog is not evenly distributed across the product portfolio. A large share of operational risk is concentrated in products that are also commercially important.

---

### 3. Current Inventory Coverage Alone Does Not Explain Historical Backlog

Inventory Days on Hand has only a **0.0016 correlation with backlog rate**, indicating essentially no linear relationship.

Backlog rates across inventory-coverage bands also remain relatively close, ranging from approximately:

- **11.53% to 12.59%**

Similarly, current stock-status groups show backlog rates between approximately:

- **11.61% and 12.35%**

This indicates that current point-in-time inventory coverage should not be treated as a direct explanation for historical backlog.

---

### 4. Raw Inventory Quantity Can Produce Misleading Relationships

Available Quantity has a positive:

- **0.5297 correlation with Backlog Quantity**
- **0.2331 correlation with Backlog Rate**

However, Average Daily Demand is also strongly associated with both inventory availability and backlog.

High-demand Product-Warehouse combinations naturally require more inventory while simultaneously generating greater backlog exposure.

Therefore, raw inventory quantity should not be interpreted independently from demand intensity.

---

### 5. Complete Fulfillment Creates a Major Delivery-Service Advantage

Among the **20,331 orders with valid delivery chronology**, 100% fulfilled orders achieve:

- **83.05% On-Time Delivery**
- **83.05% OTIF**
- **0.15 average delivery-delay days**

Partially fulfilled orders achieve only approximately:

- **56–57% On-Time Delivery**
- **0% OTIF**
- Approximately **1.9 average delay days**

This reveals a strong operational threshold: completing the order before shipment is strongly associated with substantially better downstream service performance.

---

### 6. The Presence of Backlog Matters More Than Backlog Severity

Orders with no backlog achieve:

- **83.05% On-Time Delivery**
- **0.15 average delay days**

Once backlog exists, On-Time performance falls to approximately **56–58%** across the different backlog-severity groups.

Increasing backlog percentage does not produce a consistent further deterioration in delivery performance.

The Backlog Percentage vs Delivery Delay correlation is only **0.1904**.

Therefore, the key operational distinction is whether backlog exists at all rather than simply how large the backlog percentage becomes.

---

### 7. Fulfillment and Delivery Are Related, but Not Through a Simple Linear Pattern

Order Fill Rate vs Delivery Delay has a correlation of **-0.1904**, indicating only a weak linear relationship.

However, the grouped analysis reveals a strong threshold effect at **100% fulfillment**.

This demonstrates why correlation alone should not be used to evaluate operational performance. Business relationships may be nonlinear or driven by important operational thresholds.

---

### 8. Supplier Reliability Is a Useful Procurement Performance Indicator

Supplier Reliability Factor has a **-0.2320 correlation with Supplier Delay**, indicating a weak but meaningful inverse relationship.

Supplier groups show a much clearer operational pattern:

- Reliability below 0.70 → **49.17% procurement On-Time**, **2.92 average delay days**
- Reliability 0.80–0.89 → **63.53% On-Time**, **1.03 average delay days**
- Reliability 0.90+ → **71.44% On-Time**, **0.09 average delay days**

Higher supplier reliability is therefore associated with stronger procurement delivery performance.

---

### 9. Quality Rating and Contract Lead Time Do Not Explain Supplier Delay

The analysis found:

- Quality Rating vs Supplier Delay: **0.0005**
- Contract Lead Time vs Supplier Delay: **0.0119**

Both relationships are negligible.

This indicates that the Reliability Factor is considerably more useful for differentiating actual supplier delivery performance than quality rating or contracted lead time in this dataset.

---

### 10. Historical Supplier Delay Does Not Explain Current Inventory Risk

At the Product-Warehouse level:

- Supplier Delay vs Available Quantity: **0.0234**
- Supplier Delay vs Inventory Days on Hand: **0.0368**

Both relationships are negligible.

Product-Warehouse combinations with 5+ average supplier-delay days also do not show systematically higher current stockout or reorder rates.

Therefore, the available evidence does not support treating historical supplier delay as a direct explanation for the current inventory position.

---

### 11. Procurement Is Responding to Most Replenishment Signals

There are **1,311 Product-Warehouse records requiring reorder**.

Of these, **1,080 records already have outstanding inbound PO quantity**, representing **82.38%** coverage at the record level.

Reorder-required records also show:

- **9.86 average available units**
- **13.10 average inventory days on hand**

compared with:

- **38.66 available units**
- **46.99 inventory days on hand**

for records not currently requiring reorder.

This indicates that the replenishment logic is correctly identifying lower-coverage inventory positions and that procurement is already responding to most of them.

---

### 12. Pune DC Is the Clearest Cross-Functional Warehouse Risk

Pune DC simultaneously records:

- **48.20% reorder-required rate** — highest
- **12.88% backlog rate** — joint highest
- **87.12% quantity fill rate** — joint lowest
- **70.54% On-Time Delivery** — lowest
- **45.39% OTIF** — lowest

Multiple weaknesses therefore overlap at the same warehouse.

Pune DC should be prioritized for investigation across inventory allocation, fulfillment execution, and service performance rather than treating each KPI independently.

---

### 13. Automotive Is the Weakest Fulfillment Category

Automotive records:

- **86.63% quantity fill rate**
- **13.37% backlog rate**

These are the weakest category-level fulfillment results.

Its inventory days on hand are not materially different from most other categories, suggesting that the category's weaker fulfillment performance cannot be explained simply by current inventory coverage.

---

### 14. Electronics Has the Largest Commercial Backlog Exposure

Electronics contributes:

- **28.74% of total sales**
- **28.14% of total backlog value**
- Approximately **72.42 million in backlog value**

It also holds the largest current inventory value.

Electronics is therefore not necessarily the weakest category by fulfillment rate, but it represents the largest absolute commercial exposure and should receive strong planning attention.

---

### 15. Relationship Analysis Reveals Different Types of Operational Risk

The analysis identifies three distinct patterns:

**Demand-driven risk:**  
High-demand products experience substantially greater backlog exposure.

**Execution-driven risk:**  
Incomplete fulfillment is strongly associated with weaker On-Time and OTIF performance.

**Concentration risk:**  
Backlog value is disproportionately concentrated in commercially important products.

These risks should not be addressed through a single inventory policy.

---

## Management Takeaway

The strongest evidence from the cross-functional analysis suggests that **demand concentration and incomplete fulfillment are more important indicators of service risk than current inventory coverage alone**.

Management should prioritize:

1. High-demand Product-Warehouse combinations responsible for the majority of backlog
2. High-sales-value products carrying disproportionate backlog exposure
3. Prevention of partial fulfillment before orders enter downstream shipment processes
4. Supplier reliability as a procurement-performance monitoring indicator
5. Reorder-required inventory that does not yet have adequate inbound coverage
6. Pune DC as a cross-functional operational hotspot
7. Automotive as the weakest fulfillment category
8. Electronics as the largest absolute commercial backlog exposure

Current inventory levels, stock status, and historical supplier delays should be interpreted carefully because they show weak relationships with historical backlog or current inventory outcomes.

Most importantly, these findings represent **statistical and operational associations rather than proof of causation**. The project should use them to identify where management investigation and intervention should be concentrated.

## Step 24 — Final Relationship Analysis Validation

Perform the final validation of the Relationship Analysis notebook.

The validation confirms that:

- All source datasets retain their expected dimensions
- Product-Warehouse relationship datasets reconcile with source totals
- Inventory and fulfillment integration preserves the complete inventory structure
- Delivery analysis uses only chronology-valid records
- Procurement relationship datasets reconcile correctly
- Warehouse and category cross-functional tables are complete
- Key relationship metrics are reproducible
- Consolidated findings are complete
- Step 22 reconciliation passed

This ensures that the Relationship Analysis is internally consistent and ready for final business recommendations.

In [23]:
# Final Relationship Analysis Validation

final_relationship_validation_checks = {

    # ---------------------------------------------
    # Step 22 reconciliation
    # ---------------------------------------------

    "Step 22 Reconciliation Passed":
        relationship_reconciliation_passed,


    # ---------------------------------------------
    # Core source datasets
    # ---------------------------------------------

    "Sales Orders Preserved":
        len(sales_orders) == 25_000,

    "Sales Order Items Preserved":
        len(sales_order_items) == 61_397,

    "Inventory Records Preserved":
        len(inventory) == 3_000,

    "Shipments Preserved":
        len(shipments) == 24_798,

    "Purchase Orders Preserved":
        len(purchase_orders) == 8_500,


    # ---------------------------------------------
    # Product-Warehouse relationship base
    # ---------------------------------------------

    "Product-Warehouse Fulfillment Count":
        len(product_warehouse_fulfillment) == 2_985,

    "Inventory-Fulfillment Count":
        len(inventory_fulfillment) == 3_000,

    "Inventory-Backlog Analysis Count":
        len(inventory_backlog_analysis) == 2_985,

    "Ordered Quantity Reconciles":
        np.isclose(
            inventory_fulfillment["ordered_qty"].sum(),
            501_755
        ),

    "Fulfilled Quantity Reconciles":
        np.isclose(
            inventory_fulfillment["fulfilled_qty"].sum(),
            440_576
        ),

    "Backlog Quantity Reconciles":
        np.isclose(
            inventory_fulfillment["backlog_qty"].sum(),
            61_179
        ),

    "Backlog Value Reconciles":
        np.isclose(
            inventory_fulfillment["backlog_value"].sum(),
            257_349_919.69,
            atol=0.01
        ),


    # ---------------------------------------------
    # Delivery relationships
    # ---------------------------------------------

    "Valid Fulfillment-Delivery Orders":
        len(fulfillment_delivery) == 20_331,

    "Valid Shipment Count":
        (
            shipments["chronology_valid_flag"] == "Y"
        ).sum() == 20_331,

    "Invalid Shipment Count":
        (
            shipments["chronology_valid_flag"] == "N"
        ).sum() == 4_467,

    "Fulfillment Bands Complete":
        fulfillment_delivery[
            "fulfillment_rate_band"
        ].notna().all(),

    "Backlog Severity Complete":
        fulfillment_delivery[
            "backlog_severity"
        ].notna().all(),


    # ---------------------------------------------
    # Supplier / procurement relationships
    # ---------------------------------------------

    "Valid Supplier Procurement Count":
        len(supplier_procurement_valid) == 8_470,

    "Supplier Inventory Analysis Count":
        len(supplier_inventory_valid) == 2_522,

    "Inventory Procurement Count":
        len(inventory_procurement) == 3_000,

    "Inbound PO Record Count":
        (
            inventory_procurement["inbound_po_flag"] == "Y"
        ).sum() == 2_525,

    "Outstanding PO Quantity Reconciles":
        np.isclose(
            inventory_procurement[
                "outstanding_po_qty"
            ].sum(),
            132_826
        ),


    # ---------------------------------------------
    # Cross-functional dimensions
    # ---------------------------------------------

    "Warehouse Relationship Complete":
        len(warehouse_relationship) == 6,

    "Warehouse IDs Unique":
        warehouse_relationship[
            "warehouse_id"
        ].nunique() == 6,

    "Category Relationship Complete":
        len(category_relationship) == 5,

    "Category Values Complete":
        category_relationship[
            "category"
        ].notna().all(),


    # ---------------------------------------------
    # Correlation outputs
    # ---------------------------------------------

    "Operational Correlation Matrix Complete":
        operational_correlation_matrix.shape == (10, 10),

    "Demand-Backlog Correlation Reproduced":
        np.isclose(
            operational_correlation_matrix.loc[
                "avg_daily_demand",
                "backlog_qty"
            ],
            0.7754,
            atol=0.0001
        ),

    "Inventory DOH-Backlog Correlation Reproduced":
        np.isclose(
            operational_correlation_matrix.loc[
                "inventory_days_on_hand",
                "backlog_rate_pct"
            ],
            0.0016,
            atol=0.0001
        ),

    "Supplier Reliability Correlation Reproduced":
        np.isclose(
            supplier_relationship_corr.loc[
                "reliability_factor",
                "supplier_delay_days"
            ],
            -0.2320,
            atol=0.0001
        ),


    # ---------------------------------------------
    # Findings
    # ---------------------------------------------

    "Meaningful Relationships Created":
        len(meaningful_relationships) == 10,

    "Relationship Findings Complete":
        len(relationship_findings) == 15,

    "Relationship Findings No Missing Values":
        relationship_findings.notna().all().all()
}


print("FINAL RELATIONSHIP ANALYSIS VALIDATION\n")


for check, result in final_relationship_validation_checks.items():

    print(
        f"{check:<50} : "
        f"{'PASS' if result else 'FAIL'}"
    )


final_relationship_analysis_status = (
    all(
        final_relationship_validation_checks.values()
    )
)


print(
    "\nFINAL RELATIONSHIP ANALYSIS STATUS :",
    "PASSED"
    if final_relationship_analysis_status
    else "REVIEW REQUIRED"
)

FINAL RELATIONSHIP ANALYSIS VALIDATION

Step 22 Reconciliation Passed                      : PASS
Sales Orders Preserved                             : PASS
Sales Order Items Preserved                        : PASS
Inventory Records Preserved                        : PASS
Shipments Preserved                                : PASS
Purchase Orders Preserved                          : PASS
Product-Warehouse Fulfillment Count                : PASS
Inventory-Fulfillment Count                        : PASS
Inventory-Backlog Analysis Count                   : PASS
Ordered Quantity Reconciles                        : PASS
Fulfilled Quantity Reconciles                      : PASS
Backlog Quantity Reconciles                        : PASS
Backlog Value Reconciles                           : PASS
Valid Fulfillment-Delivery Orders                  : PASS
Valid Shipment Count                               : PASS
Invalid Shipment Count                             : PASS
Fulfillment Bands Complete      